# Chuong 1: Setup & Data Loading
**Nhom 11** | Home Credit Default Risk

**Muc tieu:**
- Import thu vien
- Load 7 bang du lieu
- Aggregate & merge
- Save parquet cho cac notebook sau

In [1]:
import sys, os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import *
from src.data_loader import load_main_tables, load_auxiliary_tables, merge_all_tables
from src.utils import setup_plot_style
setup_plot_style()
try:
    import tensorflow as tf; print(f'TensorFlow : {tf.__version__}')
except: print('TensorFlow : NOT found')
import lightgbm as lgb
print(f'LightGBM   : {lgb.__version__}')
print(f'pandas     : {pd.__version__}')
print(f'Data dir   : {DATA_DIR}')
print(f'Mode       : {"FULL" if FULL_DATA else f"SAMPLE {SAMPLE_SIZE:,} rows"}')
print(f'Seed       : {SEED}')

TensorFlow : 2.21.0
LightGBM   : 4.6.0
pandas     : 3.0.3
Data dir   : d:\credisdrick\home-credit-default-risk
Mode       : SAMPLE 10,000 rows
Seed       : 42


## 1.1 Load Bang Chinh (application_train / test)

In [2]:
t0 = time.time()
app_train, app_test = load_main_tables()
print(f'\nLoad time: {time.time()-t0:.2f}s')
app_train.head(3)

[DataLoader] application_train : (10000, 122)
[DataLoader] application_test  : (48744, 121)
[DataLoader] TARGET=0 (Normal) : 9,177 (91.8%)
[DataLoader] TARGET=1 (Default): 823 (8.2%)
[DataLoader] Imbalance ratio   : 11.2:1

Load time: 5.90s


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,384575,0,Cash loans,M,Y,N,2,207000.0,465457.5,52641.0,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,1.0
1,214010,0,Cash loans,F,Y,Y,0,247500.0,1281712.5,48946.5,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,3.0
2,142232,0,Cash loans,F,Y,N,0,202500.0,495000.0,39109.5,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,3.0


## 1.2 Load 6 Bang Phu

In [3]:
valid_ids = set(app_train['SK_ID_CURR'])
bureau, bureau_bal, prev_app, installments, pos_cash, credit_card = \
    load_auxiliary_tables(valid_ids)

[DataLoader] bureau                   : (47344, 17)
[DataLoader] bureau_balance           : (200000, 3)
[DataLoader] previous_application     : (45393, 37)
[DataLoader] installments_payments    : (200000, 8)
[DataLoader] POS_CASH_balance         : (200000, 8)
[DataLoader] credit_card_balance      : (99888, 23)


## 1.3 Aggregate & Merge -> 1 Bang Du Lieu

In [4]:
t0 = time.time()
df_merged = merge_all_tables(
    app_train, bureau, bureau_bal,
    prev_app, installments, pos_cash, credit_card
)
print(f'\nMerge time: {time.time()-t0:.2f}s')
print(f'Final shape: {df_merged.shape}')

[DataLoader] Aggregating auxiliary tables...


  After merge bureau         : 136 columns
  After merge previous       : 145 columns
  After merge installments   : 153 columns
  After merge pos            : 161 columns
  After merge credit_card    : 169 columns
[DataLoader] Final shape: (10000, 169) (+47 new features)

Merge time: 15.64s
Final shape: (10000, 169)


## 1.4 Save Processed Data

In [5]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
out = PROCESSED_DIR / 'df_merged.parquet'
df_merged.to_parquet(out, index=False)
print(f'Saved: {out}')
print(f'Size : {out.stat().st_size/1024/1024:.1f} MB')
print()
print('=== Notebook 01 HOAN THANH ===')
print('Chay tiep: 02_EDA.ipynb')

Saved: d:\credisdrick\data\processed\df_merged.parquet
Size : 2.1 MB

=== Notebook 01 HOAN THANH ===
Chay tiep: 02_EDA.ipynb
